# Depth Anything V2 Small — DIMER relative depth estimation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/tutorials/depth_anything_depth_estimation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-depth--anything%2FDepth--Anything--V2--Small--hf-ffcc4d?style=flat)](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) [![Upstream](https://img.shields.io/badge/Upstream-DepthAnything%2FDepth--Anything--V2-181717?style=flat&logo=github&logoColor=white)](https://github.com/DepthAnything/Depth-Anything-V2) [![arXiv](https://img.shields.io/badge/arXiv-2406.09414-b31b1b.svg)](https://arxiv.org/abs/2406.09414)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** monocular relative depth estimation from one RGB still image using the pinned Depth Anything V2 Small weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/depth_anything_depth_estimation_pipeline/pipeline.py` at revision `e44e0575a771`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `5426e4f0f36572d16453bbda7a8389317b1bef99` (~99 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

The output is a per-pixel map of **relative inverse depth** — larger values are nearer, the scale and shift are unknown, and the values are **not metric**: they are not distances in metres and cannot be compared across images without an alignment step. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the pinned checkpoint is used as published, and the carried pipeline module adds manifest verification, input validation, resizing of the output back to the input resolution, and the `abs_rel`, `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic image drawn in code; its depth map is demonstration (plumbing) evidence, not a production-quality or benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic default input and validate it into an input manifest, run the supported task through the public API, interpret the relative-depth output and its sanity checks, understand when the shipped `abs_rel` metric applies and why the evaluation report is `not-measurable` without caller-supplied metric depth, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** metric (absolute) depth, video or temporal depth, stereo or multi-view fusion, surface normals, 3D reconstruction, or batch inference. The carried module does not provide metric depth, video, or batched paths, and this notebook must not be read as implying them.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. The model is about 99 MB and the default sample is 320 x 240 px, so a CPU runtime completes the default path (the model card records a 320 x 240 CPU prediction at 0.34 s cold on the card's workstation; a hosted CPU runtime may be slower and no figure is claimed for it).
- **Knowledge:** basic Python and NumPy, and the difference between inverse depth (larger = nearer, arbitrary scale and shift) and metric distance.
- **Data:** the default sample is a 320 x 240 RGB image drawn in this notebook, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, shorter side at least 14 px, longer side at most 4096 px, aspect ratio at most 4.0. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `depth-anything/Depth-Anything-V2-Small-hf` snapshot (~99 MB) at revision `5426e4f0f365…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'depth-anything-depth-estimation-pipeline',
    'repository_revision': 'e44e0575a7715a1d598baec34f715d11cf169700',
    'embedded_module': 'src/depth_anything_depth_estimation_pipeline/pipeline.py',
    'module_sha256': 'cab37a65cecbcbb98ec18fba6d355b02ec80f18447a9df25b56b9a0f06c50769',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/depth_anything_depth_estimation_pipeline/pipeline.py` @ `e44e0575a771`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
MODEL_REVISION = "5426e4f0f36572d16453bbda7a8389317b1bef99"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "depth-anything-v2-small"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Input ceilings. The DPT processor rescales the image so that its sides are multiples of 14 close
# to 518 px, and the raw prediction is interpolated back to the caller's resolution, so the cost that
# grows with the caller's image is the aspect ratio (backbone tokens) and the output interpolation.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 14
MAX_ASPECT_RATIO = 4.0
DEPTH_KIND = "relative"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def abs_rel(pred: np.ndarray, ref_depth: np.ndarray, *, align: bool = True) -> float:
    """Absolute relative error of a relative inverse-depth map against caller-supplied metric depth.

    The model emits relative inverse depth (disparity up to an unknown scale and shift), so the
    prediction is first aligned to ``1 / ref_depth`` by least squares over valid pixels
    (``ref_depth > 0``), inverted to depth, and scored as ``mean(|est - ref| / ref)``. With
    ``align=False`` the arrays are compared as given, which is only meaningful for metric input.
    """
    pred = np.asarray(pred, dtype=np.float64)
    ref_depth = np.asarray(ref_depth, dtype=np.float64)
    if pred.shape != ref_depth.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref_depth.shape}")
    valid = np.isfinite(ref_depth) & (ref_depth > 0) & np.isfinite(pred)
    if valid.sum() < 2:
        raise ValueError("need at least 2 valid reference pixels (ref_depth > 0)")
    if align:
        target = 1.0 / ref_depth[valid]
        design = np.stack([pred[valid], np.ones(int(valid.sum()))], axis=1)
        (scale, shift), *_ = np.linalg.lstsq(design, target, rcond=None)
        estimate = 1.0 / np.clip(scale * pred[valid] + shift, 1e-6, None)
    else:
        estimate = pred[valid]
    return float(np.mean(np.abs(estimate - ref_depth[valid]) / ref_depth[valid]))


def validate_image(image: Any) -> Image.Image:
    """Type- and size-check a caller image and return it as RGB."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    short, long = min(width, height), max(width, height)
    if short < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {short} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if long > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {long} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if long / short > MAX_ASPECT_RATIO:
        raise ValueError(f"aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",
    "short_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "long_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "aspect_ratio": [1.0, MAX_ASPECT_RATIO],
    "output": f"{DEPTH_KIND} inverse depth, float32 H x W at the input resolution (larger = nearer)",
    "preprocessing": (
        "convert to RGB; the DPT processor rescales the sides to multiples of 14 near 518 px and the "
        "raw prediction is interpolated back (bicubic) to the input resolution"
    ),
}


def validate_inputs(
    images: Any, *, names: Sequence[str] | None = None
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Each image is routed through the public ``validate_image`` that ``predict`` itself calls, so a
    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [images] if isinstance(images, Image.Image) else images
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one image is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per image")
    inputs = []
    for index, candidate in enumerate(batch):
        rgb = validate_image(candidate)
        width, height = rgb.size
        long_side, short_side = max(width, height), min(width, height)
        inputs.append(
            {
                "id": names[index] if names else f"image-{index}",
                "mode": getattr(candidate, "mode", rgb.mode),
                "size": [width, height],
                "aspect_ratio": round(long_side / short_side, 3),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "n_images": len(inputs),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_depth: Any | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_depth`` (metric depth in metres, same H x W as the prediction) the report carries
    ``abs_rel`` computed by the repository's own helper after least-squares affine alignment in inverse
    depth, as sample-sanity evidence. Without it the verdict is ``not-measurable`` and the report says
    what ground truth would make the task measurable: relative inverse depth has no intrinsic score.
    """
    depth = np.asarray(result["depth"])
    base = {
        "task": "monocular relative depth estimation",
        "score_semantics": (
            f"{result.get('depth_kind', DEPTH_KIND)} inverse depth with unknown per-image scale and shift: "
            "larger is nearer, values are not metres, carry no confidence, and no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_images": 1,
        "n_pixels": int(depth.size),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_depth is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no metric reference depth was supplied for the evaluated image",
            "needs": (
                "a metric depth map in metres with the same height and width as the image, from a depth "
                "sensor, LiDAR, or an RGB-D benchmark, scored with abs_rel(pred, ref_depth, align=True) "
                "against a constant-depth or vertical-gradient prior as the trivial baseline"
            ),
        }
    ref = np.asarray(reference_depth, dtype=np.float64)
    valid = int((np.isfinite(ref) & (ref > 0)).sum())
    return {
        **base,
        "metrics": [
            {
                "id": "abs_rel",
                "value": abs_rel(depth, ref, align=True),
                "align": True,
                "n_valid_pixels": valid,
                "estimation": (
                    "single image, least-squares affine alignment in inverse depth, no dispersion estimate"
                ),
            }
        ],
        "verdict": "sample-sanity",
        "reason": "one image with caller-supplied metric depth from the tutorial sample; not a benchmark",
        "needs": "a held-out set of metric depth maps from the deployment domain for any generalisable claim",
    }


@dataclass
class DepthAnythingPipeline:
    """Monocular relative depth estimation over the pinned Depth Anything V2 Small checkpoint."""

    _runner: Callable[[Image.Image], np.ndarray]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DepthAnythingPipeline:
        import torch
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = AutoModelForDepthEstimation.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image) -> np.ndarray:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                predicted = model(**inputs).predicted_depth
            resized = torch.nn.functional.interpolate(
                predicted.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False
            )
            return resized[0, 0].float().cpu().numpy()

        return cls(runner, resolved_device)

    def predict(self, image: Image.Image) -> dict[str, Any]:
        """Return relative inverse depth as a float32 H x W array at the input resolution."""
        rgb = validate_image(image)
        depth = np.asarray(self._runner(rgb), dtype=np.float32)
        if depth.shape != (rgb.height, rgb.width):
            raise RuntimeError(f"backend returned shape {depth.shape}, expected {(rgb.height, rgb.width)}")
        return {
            "depth": depth,
            "depth_kind": DEPTH_KIND,
            "depth_min": float(depth.min()),
            "depth_max": float(depth.max()),
            "height": rgb.height,
            "width": rgb.width,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `5426e4f0f365…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DepthAnythingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "depth-anything-v2-small",
  "modelId": "depth-anything/Depth-Anything-V2-Small-hf",
  "revision": "5426e4f0f36572d16453bbda7a8389317b1bef99",
  "files": [
    {
      "path": "README.md",
      "bytes": 4400,
      "sha256": "319566441daaac0c5ed83e75a8fd19bae0863f3275e23f147c06fe0906edf6fb"
    },
    {
      "path": "config.json",
      "bytes": 950,
      "sha256": "c56698d3643dde1f83ea2212759e6b31a22b8f827246a36dd007ee8a22b3ff75"
    },
    {
      "path": "model.safetensors",
      "bytes": 99173660,
      "sha256": "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 775,
      "sha256": "d41175c0d889477ca8fc67191e540faef14baf6275157b3fdecf78469e6bbf84"
    }
  ],
  "totalBytes": 99179785
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DepthAnythingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic sample or optional BYOD

The default sample is **synthetic**: a 320 x 240 RGB image drawn in this cell — a left-to-right luminance ramp with a bright square and a dark disc — the same kind of input the repository's smoke run used. It is generated deterministically from code (no randomness, so no seed is involved), its pixel digest is recorded in the export so a rerun can prove it saw the same input, and it needs no download and contains no personal data. It ships **no ground-truth depth**, and as a non-photographic image it lies outside the model's training distribution, so the depth map it produces is sanity evidence of the code path only — it says nothing about depth quality on real photographs and is not benchmark evidence. BYOD is optional and disabled by default. If you also hold a metric depth map for your image (a float array in metres with the same height and width, from a depth sensor or a benchmark), assign it to `reference_depth` after the upload and Section 7 will score it with the repository's `abs_rel` helper; leave it as `None` otherwise.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
SAMPLE_WIDTH = 320
SAMPLE_HEIGHT = 240

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[sample_name]))
    image.load()
    sample_kind = 'BYOD upload'
else:
    # Deterministic synthetic scene: a left-to-right luminance ramp (40 -> 220) with a bright
    # square and a dark disc. Drawn from code, so it is reproducible without any download.
    ramp = np.linspace(40, 220, SAMPLE_WIDTH, dtype=np.float32)
    rgb = np.repeat(np.repeat(ramp[None, :, None], SAMPLE_HEIGHT, axis=0), 3, axis=2).astype(np.uint8)
    image = Image.fromarray(rgb, mode='RGB')
    draw = ImageDraw.Draw(image)
    draw.rectangle([200, 60, 280, 140], fill=(245, 245, 245))
    draw.ellipse([40, 120, 130, 210], fill=(20, 20, 20))
    sample_name = f'synthetic_ramp_{SAMPLE_WIDTH}x{SAMPLE_HEIGHT}'
    sample_kind = 'synthetic'
# Metric ground-truth depth in metres with the same H x W as the image, or None. The synthetic
# sample has none, so the evaluation report is not-measurable for it.
reference_depth = None
sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'has_reference_depth': reference_depth is not None, 'pixel_sha256': sample_sha256})

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: each image is routed through the same `validate_image` that `predict` itself calls, so the checks — PIL type, shorter side at least `MIN_IMAGE_SIDE`, longer side at most `MAX_IMAGE_SIDE`, aspect ratio at most `MAX_ASPECT_RATIO` — cannot diverge between the two. It returns an **input manifest** naming the schema and ceilings, each input's identifier, observed mode, size and aspect ratio, and the verdict, written to `outputs/depth_anything_depth_estimation_input_manifest.json`. The ceilings are printed first, before any model work. To show what rejection looks like, the cell also validates a deliberately over-wide image and records the pipeline's own error message as a finding. The notebook does not crop, resize, or subsample the input; inside the pipeline the image processor rescales it so its sides are multiples of 14 near 518 px and the raw prediction is interpolated back to the input resolution, so detail finer than that internal scale is lost regardless of the source resolution. The model card records that peak accelerator memory grows with aspect ratio (812 MiB at 4096 x 1024 versus 298 MiB at 4096 x 4096 on the card's GPU), which is why an aspect ceiling exists.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_ASPECT_RATIO': MAX_ASPECT_RATIO}})
input_manifest = validate_inputs(image, names=[sample_name])
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (MIN_IMAGE_SIDE, int(MIN_IMAGE_SIDE * (MAX_ASPECT_RATIO + 1)))))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-wide-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/depth_anything_depth_estimation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Run relative depth estimation

`predict()` returns a dictionary: `depth` (float32 `H x W` at the input resolution), `depth_kind` (`"relative"`), `depth_min`, `depth_max`, `height`, `width`, `model_id`, `model_revision`. **Score semantics:** each value is relative inverse depth — larger means nearer — with an unknown per-image scale and shift. The values are not distances, not probabilities, and carry no confidence map; the pipeline applies no decision threshold, no binarisation, and no unit conversion, and any near/far cut-off is owned by the downstream caller. The checks below are falsifiable plumbing checks (shape equals the input, float32, finite, non-degenerate range) and the cell raises if any fails; they are not a quality measure. The timing is measured on the runtime identified in Section 1 for this one image.

In [ ]:
import time

started = time.perf_counter()
result = pipe.predict(image)
elapsed = time.perf_counter() - started
depth = result['depth']
checks = {
    'shape_matches_input': depth.shape == (result['height'], result['width']) == (image.height, image.width),
    'dtype_float32': depth.dtype == np.float32,
    'all_finite': bool(np.isfinite(depth).all()),
    'non_degenerate_range': result['depth_max'] > result['depth_min'],
}
if not all(checks.values()):
    raise RuntimeError(f'depth map failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'depth'})
print({'depth_shape': depth.shape, 'seconds': round(elapsed, 3), 'checks': checks})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The only metric the repository ships is `abs_rel(pred, ref_depth, align=True)` — absolute relative error `mean(|est - ref| / ref)` over pixels with `ref_depth > 0`, computed after the prediction is affinely aligned to `1 / ref_depth` by least squares and inverted to depth. It applies only when the caller supplies metric ground-truth depth of the same height and width, so with `reference_depth` set the verdict is `sample-sanity` and the metric is a single-image tutorial figure with no dispersion estimate. The synthetic sample has none, so the verdict is `not-measurable` and the report states what would make the task measurable: a metric depth map in metres from a depth sensor, LiDAR, or an RGB-D benchmark, scored against a constant-depth or vertical-gradient prior as the trivial baseline. The report is written to `outputs/depth_anything_depth_estimation_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference_depth, sample_kind=sample_kind)
with open('outputs/depth_anything_depth_estimation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric reference depth was supplied, so abs_rel is not computed; the depth map above is sanity evidence only.')

## 8. Visualize the relative depth map

The preview is a per-image min–max stretch of the relative inverse depth to 8-bit grey, shown beside the input: brighter means nearer (a larger value). The grey levels are a visual aid only — they are not distances, the stretch discards the model's scale, and two previews cannot be compared with each other. The machine-readable arrays exported in the next section, not this picture, are the outputs intended for downstream use. On the synthetic sample expect a plausible but meaningless map: the input is not a photograph.

In [ ]:
lo, hi = result['depth_min'], result['depth_max']
depth_u8 = np.round((depth - lo) / (hi - lo) * 255.0).astype(np.uint8)
depth_preview = Image.fromarray(depth_u8).convert('RGB')
side_by_side = Image.new('RGB', (image.width * 2, image.height))
side_by_side.paste(image.convert('RGB'), (0, 0))
side_by_side.paste(depth_preview, (image.width, 0))
try:
    from IPython.display import display
    display(side_by_side)
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in the next section'})

## 9. Export outputs and provenance

Four files are written under `outputs/` alongside the two role-stage artefacts: the full-resolution float32 depth array (`.npy`, the array intended for downstream use), the side-by-side preview PNG, and a JSON record that ties them to the sample identity (name, kind, pixel digest, size), the depth summary (`depth_kind`, min, max, shape, dtype, measured seconds), the input manifest, the evaluation report, the sanity checks, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, PyTorch, Transformers, device). No credentials are involved in any step, so none can reach the export.

In [ ]:
np.save('outputs/depth_anything_depth_estimation_depth.npy', depth)
side_by_side.save('outputs/depth_anything_depth_estimation_preview.png')
payload = {
    'sample': {'name': sample_name, 'kind': sample_kind, 'pixel_sha256': sample_sha256, 'width': image.width, 'height': image.height, 'has_reference_depth': reference_depth is not None},
    'prediction': {
        'depth_file': 'outputs/depth_anything_depth_estimation_depth.npy',
        'preview_file': 'outputs/depth_anything_depth_estimation_preview.png',
        'depth_kind': result['depth_kind'],
        'depth_min': result['depth_min'],
        'depth_max': result['depth_max'],
        'shape': list(depth.shape),
        'dtype': str(depth.dtype),
        'seconds': round(elapsed, 3),
    },
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sanity_checks': checks,
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/depth_anything_depth_estimation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The depth map is a model-generated map of relative inverse depth: larger is nearer, the scale and shift are unknown, and the values are not metres, not probabilities, and not comparable across images without alignment. On the synthetic default sample the map has no ground truth and the input is not a photograph, so the run demonstrates the code path and its outputs, not depth quality; the evaluation report is `not-measurable` because `abs_rel` cannot be computed without caller-supplied metric depth, and where it is computed on your own image it is a single-image tutorial figure with no dispersion estimate. The pipeline provides no confidence map, no calibrated threshold, and no metric conversion; it does not provide metric depth, video or temporal consistency, stereo/multi-view fusion, normals, 3D reconstruction, or batching. Depth quality on mirrors, glass, textureless surfaces, night scenes, fog, and non-photographic inputs is expected to degrade and is not signalled. Run-to-run variability after the deterministic sample comes from floating-point kernel selection across devices (the model card records `depth_max` 2.969 on CUDA versus 2.970 on CPU for its smoke image) and from bicubic interpolation; results on fixed hardware are repeatable but not guaranteed bitwise-identical across devices.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, accuracy on any real-image domain, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from the working-directory `weights/depth-anything-v2-small/` and rerun Section 3. A `ValueError` naming `MIN_IMAGE_SIDE`, `MAX_IMAGE_SIDE` or `MAX_ASPECT_RATIO` in Section 5 or 6: the BYOD image is outside the ceilings — resize or crop it. An out-of-memory error on a near-ceiling BYOD image: the card measured 812 MiB peak at 4096 x 1024 on its GPU; use a smaller or squarer image or a CPU runtime.

**Next experiments:** upload a real photograph with `USE_BYOD` enabled and inspect whether the near/far ordering matches the scene; if you hold a metric depth map for it, assign it to `reference_depth` and watch the report switch to `sample-sanity`, then compare `abs_rel` with `align=True` against `align=False` to see why affine alignment is required; compare the CUDA and CPU `depth_min`/`depth_max` on the same image to observe kernel-level variability.

## References

- Repository README: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf
- Upstream code: https://github.com/DepthAnything/Depth-Anything-V2
- Depth Anything V2 paper: https://arxiv.org/abs/2406.09414
- Depth Anything (V1) paper: https://arxiv.org/abs/2401.10891
- Transformers `DepthAnything` documentation: https://huggingface.co/docs/transformers/model_doc/depth_anything